# 07주 · 전반부 복습·모형 검토

수리학 CE272 · 주차별 공통 실습

핵심: 원리·식의 의미·손계산·조건 변경·결과 해석. 입력 단위와 지배 방정식·경계조건부터 확인

학습 주제: 퀴즈 #2·관수로 복습 / 중간평가

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/week07/CE272_W07.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE272-hydraulics/week07) · [계산 도구·가정](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/METHODS.md)

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)

실행: 처음에는 위에서 아래로 셀 실행. 공통 준비 셀 실행 후 주제별 실습 실행 가능

진행: 예측 → 손계산·식 설정 → 실행 → 조건 하나 변경 → 원리·가정·결과 설명

Excel 중점: 고정 f 손실의 조건별 민감도


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import importlib.util
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.integrate import trapezoid

# 로컬 공통 자료에서는 옆 과목 파일 사용. Colab에서는 같은 공개 모듈 불러오기
module_path = next((p for p in [Path.cwd()/"hydraulics.py", Path.cwd().parent/"hydraulics.py"] if p.exists()), None)
if module_path is None:
    module_path = Path(tempfile.gettempdir())/"lecture_ce272_hydraulics.py"
    urlretrieve("https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/hydraulics.py",module_path)
spec = importlib.util.spec_from_file_location("lecture_ce272_hydraulics",module_path)
hyd = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hyd)
print("수리학 공통 계산 도구 준비 완료. SI 단위·Darcy 계수 사용")

<a id="s01"></a>

## 단위·마찰계수·속도수두 점검

**먼저 예측**: 최종 유량 숫자 하나가 맞으면 에너지 방정식도 옳다고 볼 수 있는가?

**문제**: 3주차 원본 관로의 손실을 다시 계산하시오. 계산식에서 Darcy 계수와 Fanning 계수를 혼동하거나 비원형 관의 Rh를 D 대신 직접 넣는 경우의 차이를 설명하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교


In [ ]:
Q,D,L,nu,eps = .05,.2,1000.,1.004e-6,.00012
V = Q/hyd.area_pipe(D)
f_D = hyd.darcy_factor(hyd.reynolds(Q,D,nu),eps/D)
Rh = D/4
h_D = f_D*L/D*V**2/(2*hyd.G)
h_Dh = f_D*L/(4*Rh)*V**2/(2*hyd.G)
pd.Series({'correct_loss_m':h_D,'using_Dh_4Rh_m':h_Dh,
           'wrong_using_Rh_as_D_m':f_D*L/Rh*V**2/(2*hyd.G),
           'Fanning_f_for_same_flow':f_D/4})

**검토 문제**: 비원형 관의 동수직경 Dh=4A/P를 설명하시오. 다른 단면에도 마찰계수 관계를 그대로 적용하기 전에 확인할 가정도 제시하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분


<a id="s02"></a>

## 결과 검토 기록 작성

**먼저 예측**: 관의 길이·직경·유량을 바꿀 때 가장 민감한 입력은 무엇인가?

**문제**: 고정 f의 마찰손실에 대해 Q, L, D를 각각 10% 증가시킨 결과를 비교하시오. 바꾼 조건, 예상한 비율과 계산한 비율을 표로 정리하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교


In [ ]:
Q,L,D,f = .05,1000.,.2,.02
base = hyd.pipe_loss(Q,L,D,fixed_f=f)
cases=[('Q ×1.1',1.1*Q,L,D),('L ×1.1',Q,1.1*L,D),('D ×1.1',Q,L,1.1*D)]
pd.DataFrame([{'change':name,'loss_m':hyd.pipe_loss(q,l,d,fixed_f=f),
               'loss_ratio':hyd.pipe_loss(q,l,d,fixed_f=f)/base}
              for name,q,l,d in cases])

**검토 문제**: 각 비율을 Q², L, D⁻⁵의 차수와 연결하시오. f까지 다시 계산하면 위 비율이 정확히 유지되는지 검토하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분


## 학습 기록

보존식·단위·모형 범위를 근거로 잘못된 계산을 찾아 수정하시오.

선택 심화: 개념 학습 후 AI의 설명·계산 검토. 단위·가정·보존식으로 확인한 근거와 수정한 부분 기록


In [ ]:
prediction = None
hand_calculation = None
changed_condition = None
principle_and_limits = None
records = {'예측':prediction,'손계산':hand_calculation,'조건 변경':changed_condition,'원리·한계':principle_and_limits}
missing = [k for k,v in records.items() if v is None or str(v).strip() == '']
print('작성 필요:',missing) if missing else print('기록 항목 작성 완료. 계산·논리는 별도 검토 필요')